In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_csv('../results/HELMSamplesLabeled.csv')

metrics = ['stats_bleu_4','stats_f1_score','Vee','Eee','Ven','Een']

In [3]:
for m in metrics:
    if m in df:
        corr = df[[m,'label']].corr(method='spearman').iloc[0,1]
        print(f"{m:15s}  ρ={corr:.3f}")

stats_bleu_4     ρ=0.362
stats_f1_score   ρ=0.557
Vee              ρ=0.370
Eee              ρ=0.445
Ven              ρ=0.475
Een              ρ=0.625


In [4]:
results = []

for m in metrics:
    if m not in df.columns:
        continue
    
    tmp = df[['label', m]].dropna()
    if tmp[m].nunique() <= 1:
        continue  

    y = tmp['label']
    x = tmp[m].astype(float)

    auc = roc_auc_score(y, x)
    ap  = average_precision_score(y, x)

    results.append((m, auc, ap))
    print(f"{m:15s}  ROC-AUC={auc:.3f}  PR-AUC={ap:.3f}")

stats_bleu_4     ROC-AUC=0.869  PR-AUC=0.983
stats_f1_score   ROC-AUC=0.897  PR-AUC=0.970
Vee              ROC-AUC=0.760  PR-AUC=0.928
Eee              ROC-AUC=0.805  PR-AUC=0.940
Ven              ROC-AUC=0.810  PR-AUC=0.940
Een              ROC-AUC=0.839  PR-AUC=0.948


In [5]:
for m in ['Vee','Eee','Ven','Een']:
    if m in df:
        tmp = df[['label', m]].dropna()
        acc = (tmp['label'] == tmp[m]).mean()
        print(f"{m:15s}  accuracy={acc:.3f}")

Vee              accuracy=0.619
Eee              accuracy=0.734
Ven              accuracy=0.788
Een              accuracy=0.895


In [ ]:
f1_acc = []
bleu_acc = []
f1_threshold = []
bleu_threshold = []

for x in range(1000):
    SEED = x
    
    f1 = df[df['stats_f1_score'].notna()]
    min_count = f1['label'].value_counts().min()
    f1_balanced = (f1.groupby('label', group_keys=False).apply(lambda x: x.sample(min_count, random_state=SEED)).reset_index(drop=True))
    scores = f1_balanced["stats_f1_score"].values
    labels = f1_balanced["label"].values
    
    thresholds = np.sort(np.unique(scores))
    accuracies = [(labels == (scores >= t)).mean() for t in thresholds]
    
    best_idx = np.argmax(accuracies)
    best_threshold = thresholds[best_idx]
    best_accuracy = accuracies[best_idx]
    f1_acc.append(best_accuracy)
    f1_threshold.append(best_threshold)

    
    bleu = df[df['stats_bleu_4'].notna()]
    min_count = bleu['label'].value_counts().min()
    bleu_balanced = (bleu.groupby('label', group_keys=False).apply(lambda x: x.sample(min_count, random_state=SEED)).reset_index(drop=True))
    scores = bleu_balanced["stats_bleu_4"].values
    labels = bleu_balanced["label"].values
    
    thresholds = np.sort(np.unique(scores))
    accuracies = [(labels == (scores >= t)).mean() for t in thresholds]
    
    best_idx = np.argmax(accuracies)
    best_threshold = thresholds[best_idx]
    best_accuracy = accuracies[best_idx]
    bleu_acc.append(best_accuracy)
    bleu_threshold.append(best_threshold)

In [13]:
np.mean(f1_acc), np.mean(bleu_acc)

(np.float64(0.843798245614035), np.float64(0.8159027777777779))

In [15]:
np.std(f1_acc), np.std(bleu_acc)

(np.float64(0.014916797561546222), np.float64(0.029664972772074166))

In [16]:
np.mean(f1_threshold), np.mean(bleu_threshold)

(np.float64(0.3900078715037355), np.float64(0.08392882413654337))

In [17]:
np.std(f1_threshold), np.std(bleu_threshold)

(np.float64(0.11169631915761115), np.float64(0.03767258001132232))

In [20]:
df['task'].value_counts()

task
wmt_14          410
narrative_qa    300
natural_qa      300
Name: count, dtype: int64

In [24]:
narrative_qa = df.loc[df['task'] == 'narrative_qa']
natural_qa = df.loc[df['task'] == 'natural_qa']

print('narrativeQA')
for m in metrics:
    if m in narrative_qa:
        corr = narrative_qa[[m,'label']].corr(method='spearman').iloc[0,1]
        print(f"{m:15s}  ρ={corr:.3f}")

print('naturalQA')
for m in metrics:
    if m in natural_qa:
        corr = natural_qa[[m,'label']].corr(method='spearman').iloc[0,1]
        print(f"{m:15s}  ρ={corr:.3f}")

narrativeQA
stats_bleu_4     ρ=nan
stats_f1_score   ρ=0.308
Vee              ρ=0.171
Eee              ρ=0.262
Ven              ρ=0.337
Een              ρ=0.532
naturalQA
stats_bleu_4     ρ=nan
stats_f1_score   ρ=0.566
Vee              ρ=0.318
Eee              ρ=0.457
Ven              ρ=0.420
Een              ρ=0.637


In [23]:
results = []

print('narrativeQA')
for m in metrics:
    if m not in narrative_qa.columns:
        continue
    
    tmp = narrative_qa[['label', m]].dropna()
    if tmp[m].nunique() <= 1:
        continue  

    y = tmp['label']
    x = tmp[m].astype(float)

    auc = roc_auc_score(y, x)
    ap  = average_precision_score(y, x)

    results.append((m, auc, ap))
    print(f"{m:15s}  ROC-AUC={auc:.3f}  PR-AUC={ap:.3f}")

results = []

print('naturalQA')
for m in metrics:
    if m not in natural_qa.columns:
        continue
    
    tmp = natural_qa[['label', m]].dropna()
    if tmp[m].nunique() <= 1:
        continue  

    y = tmp['label']
    x = tmp[m].astype(float)

    auc = roc_auc_score(y, x)
    ap  = average_precision_score(y, x)

    results.append((m, auc, ap))
    print(f"{m:15s}  ROC-AUC={auc:.3f}  PR-AUC={ap:.3f}")

narrativeQA
stats_f1_score   ROC-AUC=0.884  PR-AUC=0.991
Vee              ROC-AUC=0.688  PR-AUC=0.969
Eee              ROC-AUC=0.798  PR-AUC=0.980
Ven              ROC-AUC=0.860  PR-AUC=0.986
Een              ROC-AUC=0.919  PR-AUC=0.992
naturalQA
stats_f1_score   ROC-AUC=0.838  PR-AUC=0.916
Vee              ROC-AUC=0.627  PR-AUC=0.754
Eee              ROC-AUC=0.738  PR-AUC=0.817
Ven              ROC-AUC=0.716  PR-AUC=0.802
Een              ROC-AUC=0.827  PR-AUC=0.864


In [25]:
print('narrativeQA')
for m in ['Vee','Eee','Ven','Een']:
    if m in narrative_qa:
        tmp = narrative_qa[['label', m]].dropna()
        acc = (tmp['label'] == tmp[m]).mean()
        print(f"{m:15s}  accuracy={acc:.3f}")

print('naturalQA')
for m in ['Vee','Eee','Ven','Een']:
    if m in natural_qa:
        tmp = natural_qa[['label', m]].dropna()
        acc = (tmp['label'] == tmp[m]).mean()
        print(f"{m:15s}  accuracy={acc:.3f}")

narrativeQA
Vee              accuracy=0.407
Eee              accuracy=0.617
Ven              accuracy=0.733
Een              accuracy=0.907
naturalQA
Vee              accuracy=0.500
Eee              accuracy=0.677
Ven              accuracy=0.647
Een              accuracy=0.833


In [ ]:
nar_accs = []
for x in range(1000):
    SEED = x 
    f1 = narrative_qa[narrative_qa['stats_f1_score'].notna()]
    min_count = f1['label'].value_counts().min()
    f1_balanced = (f1.groupby('label', group_keys=False).apply(lambda x: x.sample(min_count, random_state=SEED)).reset_index(drop=True))
    scores = f1_balanced["stats_f1_score"].values
    labels = f1_balanced["label"].values
    
    thresholds = np.sort(np.unique(scores))
    accuracies = [(labels == (scores >= t)).mean() for t in thresholds]
    
    best_idx = np.argmax(accuracies)
    nar_accs.append(accuracies[best_idx])

nat_accs = []
for x in range(1000):
    SEED = x  
    f1 = natural_qa[natural_qa['stats_f1_score'].notna()]
    min_count = f1['label'].value_counts().min()
    f1_balanced = (f1.groupby('label', group_keys=False).apply(lambda x: x.sample(min_count, random_state=SEED)).reset_index(drop=True))
    scores = f1_balanced["stats_f1_score"].values
    labels = f1_balanced["label"].values
    
    thresholds = np.sort(np.unique(scores))
    accuracies = [(labels == (scores >= t)).mean() for t in thresholds]
    
    best_idx = np.argmax(accuracies)
    nat_accs.append(accuracies[best_idx])

In [36]:
print('narrativeQA')
print(np.mean(nar_accs))
print('naturalQA')
print(np.mean(nat_accs))

narrativeQA
0.8766333333333336
naturalQA
0.800070707070707
